# Affordable Cambridge: housing data behind the website

The [Affordable Cambridge](https://affordablecambridge.org/) website credits three figures to "A Better Cambridge housing data." This notebook shows where each one comes from:

1. **−4 and +1,690:** net homes permitted by neighborhood, 2018–2025
2. **−46:** homes lost in West Cambridge when buildings with two or more homes were permitted as a single home, 1996–2024
3. **Home value:** typical Cambridge home value, January of each year, 2000–2026

All three use public data. The website's other figures link straight to their own sources.

In [1]:
from pathlib import Path
import pandas as pd

DATA_RAW = Path("../data/raw")
PERM_CSV = DATA_RAW / "cambridge_housing_permits_curated.csv"
ZHVI_CSV = DATA_RAW / "City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"

permits = pd.read_csv(PERM_CSV)
permits["Neighborhood"] = permits["Neighborhood"].str.replace(r"^\d+\s*-\s*", "", regex=True)

## 1. Net homes permitted by neighborhood, 2018–2025

**Source:** City of Cambridge, ["Housing Starts 1996-Present"](https://data.cambridgema.gov/d/6nxj-uqfe), Open Data Portal. We use a hand-curated copy (`cambridge_housing_permits_curated.csv`) that fixes duplicate and mis-attributed records in the city's export.

**Method:** for each neighborhood, add up the net change in homes across every permit from 2018 through 2025. A permit that replaces a two-family house with a six-unit building counts as +4.

The curated file leaves out one 2025 demolition, 221 Mt Auburn St (the Riverview Apartments, −63 homes) in West Cambridge. It was a one-time private demolition unrelated to housing policy, and including it would hide the pattern in the rest of West Cambridge.

In [2]:
net = (permits[permits["Year Permitted"].between(2018, 2025)]
       .groupby("Neighborhood")["Net Change in Units"].sum().astype(int)
       .sort_values(ascending=False).rename("Net homes, 2018–2025"))
net.to_frame()

,"Net homes, 2018–2025"
Neighborhood,
East Cambridge,1690
North Cambridge,833
Cambridge Highlands,675
The Port,337
Area 2/MIT,163
Neighborhood Nine,118
Baldwin,88
Wellington-Harrington,40
Cambridgeport,14


## 2. West Cambridge: homes lost to single-home conversions, 1996–2024

**Source:** the same permit records as section 1.

**Method:** find every West Cambridge permit from 1996 to 2024 where a building with two or more homes was permitted as a single home. Each one loses its prior homes minus one: a three-family house turned into a single-family house loses two homes. The table lists every such permit.

In [3]:
wc = permits[(permits["Neighborhood"] == "West Cambridge")
             & permits["Year Permitted"].between(1996, 2024)
             & (permits["Prior Residential Units"] >= 2)
             & (permits["Permitted Residential Units"] == 1)].copy()
wc["Homes lost"] = (wc["Prior Residential Units"] - 1).astype(int)
wc = wc[["Application Address", "Year Permitted", "Prior Residential Units",
         "Permitted Residential Units", "Homes lost"]].sort_values("Year Permitted").reset_index(drop=True)
wc[["Prior Residential Units", "Permitted Residential Units"]] = wc[["Prior Residential Units", "Permitted Residential Units"]].astype(int)

print(f"{len(wc)} permits, {wc['Homes lost'].sum()} homes lost")
wc

42 permits, 46 homes lost


,Application Address,Year Permitted,Prior Residential Units,Permitted Residential Units,Homes lost
0,263-265 Mt Auburn St,1997,2,1,1
1,74 Fayerweather St,2000,2,1,1
2,23 Gurney St,2003,2,1,1
3,167 Brattle St,2003,2,1,1
4,68-70 Lakeview Ave.,2003,2,1,1
5,40 Grozier Rd,2004,2,1,1
6,15 Hawthorne Park,2005,2,1,1
7,19 Ash St,2005,3,1,2
8,112 Lexington Ave,2006,2,1,1
9,36 Royal Ave,2007,3,1,2


## 3. Typical Cambridge home value, 2000–2026

**Source:** [Zillow Research](https://www.zillow.com/research/data/), Zillow Home Value Index (ZHVI) for Cambridge, MA: single-family homes and condos, middle price tier (homes between the 33rd and 67th percentile), smoothed and seasonally adjusted. Values are in nominal dollars, not adjusted for inflation.

**Method:** take the January value for each year.

Zillow revises past values each month, so a newer download may differ slightly. The copy here was downloaded in May 2026.

In [4]:
zhvi = pd.read_csv(ZHVI_CSV)
cambridge = zhvi[(zhvi.RegionName == "Cambridge") & (zhvi.State == "MA")].iloc[0]

home_value = pd.DataFrame({"Year": range(2000, 2027)})
home_value["Typical home value"] = [round(cambridge[f"{y}-01-31"], -3) for y in home_value.Year]
home_value["Typical home value"] = home_value["Typical home value"].map("${:,.0f}".format)

ratio = cambridge["2026-01-31"] / cambridge["2000-01-31"]
print(f"January 2026 is {ratio:.1f} times January 2000")
home_value

January 2026 is 3.3 times January 2000


,Year,Typical home value
0,2000,"$314,000"
1,2001,"$373,000"
2,2002,"$421,000"
3,2003,"$455,000"
4,2004,"$473,000"
5,2005,"$505,000"
6,2006,"$517,000"
7,2007,"$494,000"
8,2008,"$491,000"
9,2009,"$479,000"
